In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.format("parquet").load("abfss://bronze@databricksgvse2e.dfs.core.windows.net/products")
df.show(3)

In [0]:
df =df.drop("_rescued_data")
display(df)


In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat.bronze.upper_data(brand STRING)
  RETURNS STRING
  LANGUAGE SQL
  RETURN upper(brand)

In [0]:
%sql
CREATE OR REPLACE FUNCTION databricks_cat.bronze.discount(price DOUBLE, discount DOUBLE)
  RETURNS DOUBLE
  LANGUAGE PYTHON
  AS $$
       return price - (price * discount/100)
  $$

In [0]:
df_price = df.withColumn("discounted_price", expr("ROUND(databricks_cat.bronze.discount(price,20),2)"))
display(df_price)

In [0]:
df_final = df_price.withColumn("brand", expr("databricks_cat.bronze.upper_data(brand)"))
display(df_final)


In [0]:
    df_final = df_final.withColumn("updated_timestamp", current_timestamp())
    df_final.write.mode("overwrite").format("delta").option("mergeSchema", "true").save("abfss://silver@databricksgvse2e.dfs.core.windows.net/products")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS databricks_cat.silver.products
USING DELTA
LOCATION 'abfss://silver@databricksgvse2e.dfs.core.windows.net/products'

In [0]:
DF=spark.read.table("databricks_cat.silver.products")
display(DF)